# 01 — Charger et contrôler les données v2

Contribution partielle à **L03** et **L12**. Objectif : comprendre le passage des sources aux tableaux en mémoire, puis vérifier les effectifs sans afficher les textes protégés.

Un **manifeste** est une liste de références : comme une table d'index en base de données, il dit quelles lignes sélectionner et dans quelle partition. Un **DataFrame** est le tableau chargé en RAM à partir de ces références ; nous n'en sauvegardons pas ici une copie sur disque.

Prérequis : copie locale du dépôt, Python/uv installés et `sh scripts/project_uv.sh sync --locked` déjà réussi (guide : `docs/20_environnement_python.md`). Le noyau du notebook ne nécessite que la bibliothèque standard : les traitements utilisent le Python verrouillé du projet dans un sous-processus, hors ligne. Un échec de cache ou d'installation doit être résolu avant la relance ; aucune installation automatique n'est faite ici.

Par défaut nous utilisons quatre commentaires **inventés**, pas les corpus. Pour le mode réel, les sources autorisées doivent déjà être extraites dans les dossiers locaux attendus. Aucun téléchargement, nouvelle décision humaine, modification d'original ou accès aux lignes du test n'est demandé. Kaggle : dépôt privé manuel et adaptation des chemins encore à valider.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'scripts/project_uv.sh').is_file()), None)
if ROOT is None:
    raise RuntimeError('Ouvrir le notebook depuis le dépôt ou son dossier notebooks/.')
sys.path.insert(0, str(ROOT / 'scripts'))
from notebook_support import run_project

# Remplacer par 'real' pour activer explicitement les sources autorisées.
DATA_MODE = os.environ.get('TUNISCOPE_DATA_MODE', 'synthetic')
if DATA_MODE not in {'synthetic', 'real'}:
    raise ValueError('DATA_MODE doit être synthetic ou real')
print('Mode données :', DATA_MODE)

Mode données : synthetic


## Sources → manifestes → RAM → normalisation

En mode réel, `load_working_data.py` sélectionne explicitement **v2**, contrôle les empreintes des sources/manifestes et charge train/validation. Train sert à apprendre, validation à comparer ; le test est réservé à l'évaluation finale. Un **groupe** est une famille indivisible pour le partitionnement : ses lignes ne doivent pas être réparties entre train et validation. Cela ne signifie pas qu'elles portent toutes le même label.

`normalize_text.py` crée une copie avec `text_norm` : Unicode NFC et espaces regroupés seulement. `text_raw` reste intact. Négation, chiffres Arabizi, emojis et répétitions sont conservés. `as_pandas` transforme cette copie en tableau en RAM. Nous n'affichons que sa forme et des comptes agrégés.

AR/LATIN/MIXED/UNKNOWN indiquent la présence de lettres Unicode, **pas une langue détectée**. Aucun tokenizer neuronal n'est exécuté.

In [2]:
overview = json.loads(run_project('notebook_support.py', '--mode', DATA_MODE, capture=True))
print(json.dumps(overview, ensure_ascii=False, indent=2))

{
  "corpora": {
    "invented": {
      "columns": [
        "id",
        "text_raw",
        "text_norm",
        "task",
        "label",
        "script",
        "languages",
        "source",
        "source_id",
        "source_file",
        "source_line",
        "group_id",
        "split",
        "official_split",
        "annotation_status",
        "license",
        "normalization_version"
      ],
      "dataframe_shape": [
        4,
        17
      ],
      "partitions": {
        "train": {
          "groups": 2,
          "labels": {
            "NEG": 1,
            "POS": 1
          },
          "normalization_changed_rows": 1,
          "rows": 2,
          "writing_system": {
            "LATIN": 2
          }
        },
        "validation": {
          "groups": 2,
          "labels": {
            "NEG": 1,
            "POS": 1
          },
          "normalization_changed_rows": 0,
          "rows": 2,
          "writing_system": {
            "MIXED": 2


In [3]:
expected = {'tsac': (10053, 1774), 'madar': (6388, 800)}
for corpus, summary in overview['corpora'].items():
    parts = summary['partitions']
    assert summary['selected_test_rows'] == 0
    assert summary['shared_train_validation_groups'] == 0
    assert summary['raw_input_preserved']
    assert summary['dataframe_shape'] == [sum(p['rows'] for p in parts.values()), 17]
    assert all(sum(p['labels'].values()) == p['rows'] for p in parts.values())
    if DATA_MODE == 'real':
        assert tuple(parts[s]['rows'] for s in ('train', 'validation')) == expected[corpus]
print('Contrôles réussis : comptes, groupes séparés, texte brut préservé, aucune ligne test sélectionnée.')

Contrôles réussis : comptes, groupes séparés, texte brut préservé, aucune ligne test sélectionnée.


## Interprétation et limites

En mode fictif, une réussite prouve seulement que le chemin logiciel fonctionne. En mode réel, les formes attendues sont TSAC **11 827 × 17** et MADAR **7 188 × 17**, train et validation réunis. Ces comptes ne sont pas des scores de modèle. Les champs `script`/`languages` du chargeur restent UNKNOWN/und ; les comptes d'écriture sont calculés séparément pour l'audit.

Aucun test officiel n'est sélectionné ni évalué ici. Les fichiers physiques MADAR comprennent plusieurs partitions ; leur lecture/vérification d'intégrité ne signifie pas entraînement sur leurs lignes test. Ce notebook réutilise les manifestes figés et ne reconstitue pas les revues humaines privées qui les ont précédés.

L03 reste partiel : acquisition documentée, politique de préparation et statistiques existent, mais tokenisation neuronale et relance cloud restent à vérifier. Voir `docs/31_notebooks_reproductibles.md`. Passer ensuite à `02_baselines_classiques.ipynb`.